# DS4400 Final Project - Predicting Social Media Engagement Using Lifestyle and Behavioral Data
### Isabel Taylor, Elaina Kreher
Project link: https://github.com/izzietaylor123/DS4400_final_project 

### Standardization of data 

In [14]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv('instagram_usage_lifestyle.csv')

# isolate the target variable (user_engagement_score) and drop it from the dataframe
user_engagement_score = df['user_engagement_score'] 
df.drop('user_engagement_score', axis=1, inplace=True)

# drop other non-impactful features
df.drop('user_id', axis=1, inplace=True)
df.drop('app_name', axis=1, inplace=True) # all users use Instagram, so this does not help

# convert categorical variables to numerical variables when binary or ordinal
df['income_level'] = df['income_level'].map({'Low': 0, 'Lower-middle': 1, 'Middle': 2, 'Upper-middle': 3, 'High': 4})
df['has_children'] = df['has_children'].map({'No': 0, 'Yes': 1})
df['diet_quality'] = df['diet_quality'].map({'Very poor': 0, 'Poor': 1, 'Average': 2, 'Good': 3, 'Excellent': 4})
df['smoking'] = df['smoking'].map({'No': 0, 'Former': 1, 'Yes': 2})
df['alcohol_frequency'] = df['alcohol_frequency'].map({'Never': 0, 'Rarely': 1, 'Weekly': 2, 'Several times a week': 3, 'Daily': 4})
df['uses_premium_features'] = df['uses_premium_features'].map({'No': 0, 'Yes': 1})
df['privacy_setting_level'] = df['privacy_setting_level'].map({'Private': 0, 'Friends only': 1, 'Public': 2})
df['two_factor_auth_enabled'] = df['two_factor_auth_enabled'].map({'No': 0, 'Yes': 1})
df['biometric_login_used'] = df['biometric_login_used'].map({'No': 0, 'Yes': 1})
df['subscription_status'] = df['subscription_status'].map({'Free': 0, 'Premium': 1, 'Business': 2})

# convert categorical variables to numerical variables when nominal (one-hot encoding)
df = pd.get_dummies(df, columns=['gender'], drop_first=True) # drop_first=True to avoid multicollinearity
df = pd.get_dummies(df, columns=['country'], drop_first=True)
df = pd.get_dummies(df, columns=['urban_rural'], drop_first=True)
df = pd.get_dummies(df, columns=['employment_status'], drop_first=True)
df = pd.get_dummies(df, columns=['education_level'], drop_first=True)
df = pd.get_dummies(df, columns=['relationship_status'], drop_first=True)
df = pd.get_dummies(df, columns=['content_type_preference'], drop_first=True)
df = pd.get_dummies(df, columns=['preferred_content_theme'], drop_first=True)

# convert last login date to days since last login (replace string in format 'YYYY-MM-DD' with number of days since that date)
df['last_login_date'] = pd.to_datetime(df['last_login_date'], format='%Y-%m-%d', errors='coerce')
today = pd.Timestamp('2026-03-01') # use a fixed date for "today" to ensure consistency in the dataset instead of using pd.Timestamp.now() which would change every time the code is run
df['days_since_last_login'] = (today - df['last_login_date']).dt.days
df.drop(columns=['last_login_date'], inplace=True)

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df, user_engagement_score, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# There are no missing values in the dataset, so we do not need to deal with NaN values

df.head()

,age,income_level,has_children,exercise_hours_per_week,sleep_hours_per_night,diet_quality,smoking,alcohol_frequency,perceived_stress_score,self_reported_happiness,...,content_type_preference_Stories,content_type_preference_Videos,preferred_content_theme_Fashion,preferred_content_theme_Fitness,preferred_content_theme_Food,preferred_content_theme_Music,preferred_content_theme_Other,preferred_content_theme_Tech,preferred_content_theme_Travel,days_since_last_login
0,51,4,0,7.2,7.7,3,0,1,3,8,...,False,False,False,False,False,False,False,True,False,119
1,64,2,0,10.9,8.6,0,0,1,1,1,...,False,False,True,False,False,False,False,False,False,344
2,41,2,0,5.0,6.7,3,0,1,4,10,...,False,False,False,False,False,False,True,False,False,203
3,27,2,0,10.6,6.5,1,2,0,18,1,...,True,False,False,False,False,False,False,True,False,335
4,55,3,0,7.7,6.8,2,0,0,19,1,...,False,True,False,False,True,False,False,False,False,347


In [15]:
print("Summary statistics for user engagement score:")

max_engagement_score = user_engagement_score.max()
print(f'Max Engagement Score: {max_engagement_score}')
min_engagement_score = user_engagement_score.min()
print(f'Min Engagement Score: {min_engagement_score}')
mean_engagement_score = user_engagement_score.mean()
print(f'Mean Engagement Score: {mean_engagement_score}')
median_engagement_score = user_engagement_score.median()
print(f'Median Engagement Score: {median_engagement_score}')

Summary statistics for user engagement score:
Max Engagement Score: 18.67
Min Engagement Score: 0.67
Mean Engagement Score: 1.6446420883573574
Median Engagement Score: 1.1


### Model 1: Linear Regression Model

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

# create and fit the linear regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# make predictions on the test set
y_pred = lr_model.predict(X_test)
y_train_pred = lr_model.predict(X_train)

# evaluate the model
mse_test = np.sqrt(mean_squared_error(y_test, y_pred))
mse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
r2_test = r2_score(y_test, y_pred)
r2_train = r2_score(y_train, y_train_pred)

print(f'Root Mean Squared Error (Test): {mse_test}')
print(f'Root Mean Squared Error (Train): {mse_train}')
print(f'Mean Absolute Error (Test): {mean_absolute_error(y_test, y_pred)}')
print(f'Mean Absolute Error (Train): {mean_absolute_error(y_train, y_train_pred)}')
print(f'R^2 Score (Test): {r2_test}')
print(f'R^2 Score (Train): {r2_train}')


Root Mean Squared Error (Test): 1.2083038173645568
Root Mean Squared Error (Train): 1.2041187062620065
Mean Absolute Error (Test): 0.8288197791269428
Mean Absolute Error (Train): 0.8270707469105301
R^2 Score (Test): 0.5567397301109961
R^2 Score (Train): 0.5584728266201094


#### Linear Regression Model Analysis
The model does not perform well
- Test R² ≈ 0.55 --> the model can only explain about 55% of the variation in the engagement score

Prediction error is relatively high
- The MSE (both test and train) ≈ 1.2 --> means the model’s predictions are usually only about 1.2 units away from the true value on average, when our predicitons fall on a scale of 1-18 (with the average score being around 1.6), this is a VERY large error margin

The model likely isn’t overfitting
- The test R² (0.5567) is almost the same as the train R² (0.5584) (when these numbers are close it means the model generalizes well to new data)

Linear regression's low performance makes sense on our data set, as there are many non-linear relationships between features and the target. We have many one-hot encoded features that may not be linearly correlated with the target (ie gender or country), which will throw off the linear regression model. 

In [17]:
lr_coefs = []
for i, coef in enumerate(lr_model.coef_):
    lr_coefs.append((df.columns[i], coef))

lr_coefs_df = pd.DataFrame(lr_coefs, columns=["Feature", "Coefficient"])
lr_coefs_df = lr_coefs_df.sort_values(by="Coefficient", ascending=False)
print("\nLinear Regression Coefficients:")
display(lr_coefs_df)


Linear Regression Coefficients:


,Feature,Coefficient
20,daily_active_minutes_instagram,0.647237
25,likes_given_per_day,0.431841
22,posts_created_per_week,0.297005
26,comments_written_per_day,0.259469
9,self_reported_happiness,0.151128
...,...,...
8,perceived_stress_score,-0.122634
23,reels_watched_per_day,-0.184188
24,stories_viewed_per_day,-0.850028
40,average_session_length_minutes,-1.280666


### Model 2: Random Forest Regressor
This model is a good next step after linear regression because it can learn nonlinear relationships and interactions between features

Important note: Random forests do not require feature scaling but notebook already standardized X_train and X_test

In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [19]:
# Creating a small validation set from the training data

# Keep the original X_test / y_test untouched so that the final test score stays fair and unbiased
# Because this dataset is very large I tune on a random sample of the training set instead of the full training set

rng = np.random.default_rng(42)

# Chose a manageable random sample from the training data for tuning (20000)
tune_size = min(20000, len(X_train))
tune_indices = rng.choice(len(X_train), size=tune_size, replace=False)

X_tune = X_train[tune_indices]
y_tune = y_train.iloc[tune_indices]

# Split that sample into a smaller training portion and a validation portion
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_tune, y_tune, test_size=0.15, random_state=42
)

In [20]:
# Testing some hyperparameter settings

# To avoid overfitting: limiting tree depth and require each leaf to contain several samples (makes the forest more generalizable)

# max_depth: limits how deep each tree can grow
# min_samples_leaf: requires more than 1 sample in each leaf, which smooths predictions
# max_features='sqrt': each tree only sees a subset of features at each split which reduces correlation between trees and helps prevent overfitting

param_grid = [
    {"max_depth": 10, "min_samples_leaf": 5, "max_features": "sqrt"},
    {"max_depth": 15, "min_samples_leaf": 5, "max_features": "sqrt"},
    {"max_depth": 20, "min_samples_leaf": 5, "max_features": "sqrt"},
    {"max_depth": 15, "min_samples_leaf": 10, "max_features": "sqrt"},
    {"max_depth": 20, "min_samples_leaf": 10, "max_features": "sqrt"},
]

results = []

for params in param_grid:
    # Build one candidate random forest
    candidate_rf = RandomForestRegressor(
        n_estimators=60,          # enough trees for stability during tuning
        random_state=42,
        n_jobs=-1,                # use all CPU cores available
        bootstrap=True,           # needed for out-of-bag scoring
        oob_score=True,           # extra generalization check
        **params
    )

    # Fit on the sub-training split
    candidate_rf.fit(X_subtrain, y_subtrain)

    # Predict on the validation split
    val_pred = candidate_rf.predict(X_val)

    # Measure validation performance
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    val_r2 = r2_score(y_val, val_pred)

    # Store the results so we can compare candidates
    results.append({
        "params": params,
        "validation_rmse": val_rmse,
        "validation_r2": val_r2,
        "oob_r2": candidate_rf.oob_score_
    })

results_df = pd.DataFrame(results).sort_values(by="validation_rmse")
print("Validation results (lower RMSE is better):")
display(results_df)

# Pick the hyperparameters with the best validation RMSE
best_params = results_df.iloc[0]["params"]
print("Best hyperparameters:", best_params)

Validation results (lower RMSE is better):


,params,validation_rmse,validation_r2,oob_r2
2,"{'max_depth': 20, 'min_samples_leaf': 5, 'max_...",0.314207,0.970417,0.972261
1,"{'max_depth': 15, 'min_samples_leaf': 5, 'max_...",0.314353,0.970389,0.971513
4,"{'max_depth': 20, 'min_samples_leaf': 10, 'max...",0.322789,0.968779,0.969025
0,"{'max_depth': 10, 'min_samples_leaf': 5, 'max_...",0.325396,0.968272,0.969761
3,"{'max_depth': 15, 'min_samples_leaf': 10, 'max...",0.325648,0.968223,0.968960


Best hyperparameters: {'max_depth': 20, 'min_samples_leaf': 5, 'max_features': 'sqrt'}


#### Analysis
The model performs very well
- Validation R² ≈ 0.97 --> the model explains about 97% of the variation in the engagement score

Prediction error is low
- The RMSE ≈ 0.314 --> means the model’s predictions are usually only about 0.31 units away from the true value on average

The model likely isn’t overfitting
- The OOB R² (0.972) is almost the same as the validation R² (0.970) (when these numbers are close it means the model generalizes well to new data)

The best model configuration was:
- max_depth = 20 → trees can capture complex patterns
- min_samples_leaf = 5 → prevents trees from memorizing data
- max_features = sqrt → improves randomness and reduces overfitting

Multiple parameter combinations performed similarly
- The top few models have almost identical RMSE and R² values --> means the model is stable and reliable

In [21]:
# Training the final random forest on the full training set
# Using more trees here than during tuning to make the final model more stable

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    bootstrap=True,
    oob_score=True,
    **best_params
)

rf_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, max_features='sqrt', min_samples_leaf=5,
                      n_jobs=-1, oob_score=True, random_state=42)

In [22]:
# Evaluating on both train and test sets

# To save time since the dataset is huge I'm evaluating training performance on a random sample of the training set

train_eval_size = min(20000, len(X_train))
train_eval_indices = rng.choice(len(X_train), size=train_eval_size, replace=False)

X_train_eval = X_train[train_eval_indices]
y_train_eval = y_train.iloc[train_eval_indices]

# Predictions
train_pred = rf_model.predict(X_train_eval)
test_pred = rf_model.predict(X_test)

# Metrics
train_rmse = np.sqrt(mean_squared_error(y_train_eval, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
train_r2 = r2_score(y_train_eval, train_pred)
test_r2 = r2_score(y_test, test_pred)
oob_r2 = rf_model.oob_score_

print("\nRandom Forest Results")
print(f"Train RMSE (sample): {train_rmse:.4f}")
print(f"Test RMSE:           {test_rmse:.4f}")
print(f"Train R^2 (sample):  {train_r2:.4f}")
print(f"Test R^2:            {test_r2:.4f}")
print(f"OOB R^2:             {oob_r2:.4f}")


Random Forest Results
Train RMSE (sample): 0.1429
Test RMSE:           0.1842
Train R^2 (sample):  0.9937
Test R^2:            0.9897
OOB R^2:             0.9893


#### Analysis
Test R² = 0.9897 (≈ 0.99)
- Model explains about 99% of the variation in engagement score on completely unseen data --> model predicts engagement accurately

Train R² = 0.9937 vs Test R² = 0.9897
- These numbers are very close (a huge gap would mean overfitting)


Train RMSE = 0.1429
Test RMSE = 0.1842
The prediction error only increased slightly on new data which is normal and shows good generalization

OOB R² = 0.9893
- Out-of-bag scoring is an internal cross-validation used by random forests
- It being almost identical to the test R² confirms the model is stable and not memorizing the training data

In [23]:
# Checking for overfitting

r2_gap = train_r2 - test_r2
rmse_gap = test_rmse - train_rmse

print("\nOverfitting check")
print(f"R^2 gap (train - test): {r2_gap:.4f}")
print(f"RMSE gap (test - train): {rmse_gap:.4f}")

if r2_gap < 0.03:
    print("This gap is small, so the random forest does not appear to be heavily overfitted.")
else:
    print("This gap is fairly large, so the model may be overfitting. Try reducing max_depth or increasing min_samples_leaf.")


Overfitting check
R^2 gap (train - test): 0.0040
RMSE gap (test - train): 0.0412
This gap is small, so the random forest does not appear to be heavily overfitted.


In [24]:
# Showing the most important features

# Recover column names from the dataframe that existed before scaling
feature_names = df.columns

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nTop 15 most important features:")
display(feature_importance_df.head(15))


Top 15 most important features:


,feature,importance
34,time_on_reels_per_day,0.166522
31,time_on_feed_per_day,0.147950
40,average_session_length_minutes,0.118580
20,daily_active_minutes_instagram,0.080312
33,time_on_messages_per_day,0.080248
25,likes_given_per_day,0.062667
21,sessions_per_day,0.057053
24,stories_viewed_per_day,0.054620
32,time_on_explore_per_day,0.052216
23,reels_watched_per_day,0.042823


### Model 3: K-Nearest Neighbors

In [25]:
from sklearn.neighbors import KNeighborsRegressor

# our dataset is very large and takes >30 minutes to run one KNN on it, so we will run KNN on a random sample of the training data instead of the full training data to save time 
rng = np.random.default_rng(42)
sample_size = min(80000, len(X_train))
sample_indices = rng.choice(len(X_train), size=sample_size, replace=False)
X_sample = X_train[sample_indices]
y_sample = y_train.iloc[sample_indices]

best_knn_model = None
best_test_r2 = 0
results = []

for k in range(3, 27, 2): # trying different k to find the best one (lower k can overfit, higher k can underfit)

    knn_model = KNeighborsRegressor(n_neighbors=k, algorithm='brute') # brute-force algorithm is more efficient for large datasets and high-dimensional data
    knn_model.fit(X_sample, y_sample)

    y_pred_knn = knn_model.predict(X_test)
    y_pred_knn_train = knn_model.predict(X_sample)

    test_r2 = r2_score(y_test, y_pred_knn)

    if test_r2 > best_test_r2:
        best_test_r2 = test_r2
        best_knn_model = knn_model
    results.append((k, np.sqrt(mean_squared_error(y_test, y_pred_knn)), np.sqrt(mean_squared_error(y_sample, y_pred_knn_train)), test_r2, r2_score(y_sample, y_pred_knn_train)))

results_df = pd.DataFrame(results, columns=["k value", "test_rmse", "train_rmse", "test_r2", "train_r2"])
print("\nKNN results:")
display(results_df)


KNN results:


,k value,test_rmse,train_rmse,test_r2,train_r2
0,3,1.174476,0.849947,0.581211,0.782664
1,5,1.092515,0.898673,0.637622,0.757031
2,7,1.057480,0.921223,0.660492,0.744685
3,9,1.040920,0.937144,0.671041,0.735783
4,11,1.031579,0.947657,0.676919,0.729822
5,13,1.025887,0.954640,0.680474,0.725826
6,15,1.021659,0.960766,0.683103,0.722296
7,17,1.020027,0.966100,0.684114,0.719204
8,19,1.018820,0.970506,0.684862,0.716637
9,21,1.018248,0.975503,0.685215,0.713711


#### Analysis
Test R² = 0.581 (k=3) -> 0.685 (k=25)
- Model explains slightly more than half of the variation in engagement score on completely unseen data --> model does not predict engagement very accurately
- Increase in K (number of neighbors) does improve this R² score, but it never reaches a good score (compared to our previous models).

Test R² = 0.581211 vs Train R² = 0.782664 (at k=3)
Test R² = 0.685183 vs Train R² = 0.708129 (at k=25)
- Test and train R² were starkly different at k = 3, indicating overfitting, but the gap decreases as k increased.

Train RMSE = 0.849 (k=3) -> 0.984 (k=25)
Test RMSE = 1.174 (k=3) -> 1.018 (k=25)
- These are very high RMSE scores, as they mean the prediction is often around 1 unit off from the true answer. Since our target values fall between 1 and 18 (and are typically around 1.6), this is a very significant error.

Also, due to the naive approach nature of KNN, we could not train it on the full training set (it took over an hour to run), so we had to use a smaller sample. Since we did this randomly, we got as good of a representative sample as possible, but it is still a disadvantage for the model to train only on a smaller subset of the training data. 

The KNN model struggles to predict our data because we had highly dimensional data and around a million rows of data. KNN has dificulty with large numbers of features, which is likely the reason the R^2 scores were so low. While the training R² was originally decent (0.78 at k=3), the test R² was 0.58, indicating significant overfitting and showing the model was not accurately prediciting when given new data. As k increased, the training R² decreased, but testing R² incrased, so the gap between them lessened, indicating less overfitting. Overall, a higher k value meant better model test performance and less overfitting, but even the model's best scores did not compare with the Random Forest Regressor. 

In [ ]:
# commented out to reduce runtime of notebook, since this takes an extremely long time to run

# from sklearn.inspection import permutation_importance

# knn_importance = permutation_importance(
#     best_knn_model,
#     X_test,
#     y_test,
#     n_repeats=3,
#     random_state=42,
#     scoring="r2"
#  )

# knn_importance_df = pd.DataFrame({
#     "feature": df.columns,
#     "importance": knn_importance.importances_mean,
#     "std": knn_importance.importances_std
# }).sort_values(by="importance", ascending=False)

# print("\nK-Nearest Neighbors Permutation Importance:")
# display(knn_importance_df.head(15))

Since KNN is a non-parametric model, it does not have coefficients the way some of the other models do, so we used permutation importance to give similar metrics

One other solution to KNN not being able to deal with high-dimensional data is to reduce the number of features. Using Principal Component Analysis (PCA), we can reduce the dimenstionality of the data by transforming our many features into a smaller set of new uncorrelated variables called principal components, which capture the most variance in the data. Typically, this is done with standardization, but since our data was already standardized in the full set, we can skip that step here.

In [ ]:
# this takes about 40 minutes to run, comment out if you want to save time

# using PCA to reduce dimensionality 
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsRegressor # some of these imports are repeated so we can run cells independently to save time 
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

pca_nums = [5, 8, 10] # trying different numbers of PCA components to find the best one (lower number can underfit, higher number can overfit)
pca_results = []

best_pca_model = None
best_pca_transformer = None
best_pca_test_r2 = 0

for n in pca_nums:
    pca = PCA(n_components=n) # reducing to n components
    X_pca = pca.fit_transform(X_train)

    knn_model_pca = KNeighborsRegressor(n_neighbors=25, algorithm='brute') # using the best k value from the previous tuning step, but now with PCA features instead of the original features
    knn_model_pca.fit(X_pca, y_train)

    y_pred_knn_pca = knn_model_pca.predict(pca.transform(X_test))
    y_pred_knn_pca_train = knn_model_pca.predict(X_pca)

    test_r2_pca = r2_score(y_test, y_pred_knn_pca)

    if test_r2_pca > best_pca_test_r2:
        best_pca_test_r2 = test_r2_pca
        best_pca_model = knn_model_pca
        best_pca_transformer = pca

    pca_results.append((n, np.sqrt(mean_squared_error(y_test, y_pred_knn_pca)), np.sqrt(mean_squared_error(y_train, y_pred_knn_pca_train)), test_r2_pca, r2_score(y_train, y_pred_knn_pca_train), mean_absolute_error(y_test, y_pred_knn_pca), mean_absolute_error(y_train, y_pred_knn_pca_train)))


pca_results_df = pd.DataFrame(pca_results, columns=["num components", "test_rmse", "train_rmse", "test_r2", "train_r2", "test_mae", "train_mae"])
print("\nKNN with PCA results:")
display(pca_results_df)


KNN with PCA results:


,num components,test_rmse,train_rmse,test_r2,train_r2,test_mae,train_mae
0,5,0.712771,0.682199,0.845757,0.858277,0.251855,0.241598
1,8,0.702753,0.675316,0.850062,0.861122,0.252047,0.242652
2,10,0.735492,0.712908,0.835766,0.845230,0.266610,0.258178


#### Analysis

Test R²: 0.8471
- The model can predict 85% of the variance — this is a much better score than before using PCA.

Train R²: 0.8589
- Very close train and test R² scores do not indicate overfitting. 

Test RMSE: 0.7097, Train RMSE: 0.6806
- Both of these are much lower than the pre-PCA model, and are also pretty similar (indicating not overfitting).

Overall, the KNN model, both with and without PCA, is not well suited to our data set and siginificantly underperforms when compared to Random Forest Regressor. Using PCA to limit the number of features was helpful, and improved metrics significantly compared to the non-PCA, both because there were less features for the model to worry about, but also because using PCA allowed us to train on the entire training set, instead of a subset. However, it was still not our best-performing model. Due to the high-dimensional characteristics of or dataset, it makes sense that this is not the most fitting model for our data set, but it does give us insight into how well our other models can perform by giving us a low baseline. 

In [28]:
# commented out to reduce runtime of notebook, since this takes an extremely long time to run

# from sklearn.inspection import permutation_importance

# pca_test_features = best_pca_transformer.transform(X_test)

# pca_importance = permutation_importance(
#     best_pca_model,
#     pca_test_features,
#     y_test,
#     n_repeats=3,
#     random_state=42,
#     scoring="r2"
#  )

# pca_importance_df = pd.DataFrame({
#     "principal_component": [f"PC{i + 1}" for i in range(len(pca_importance.importances_mean))],
#     "importance": pca_importance.importances_mean,
#     "std": pca_importance.importances_std
# }).sort_values(by="importance", ascending=False)

# print("\nKNN with PCA Permutation Importance (principal components):")
# display(pca_importance_df.head(10))

# # Show which original features contribute most to the leading principal components
# loadings = pd.DataFrame(
#     best_pca_transformer.components_.T,
#     index=df.columns,
#     columns=[f"PC{i + 1}" for i in range(best_pca_transformer.n_components_)]
#  )

# print("\nTop original-feature loadings for the most important PCA components:")
# for component in pca_importance_df.head(3)["principal_component"]:
#     top_features = loadings[component].abs().sort_values(ascending=False).head(10)
#     print(f"\n{component}:")
#     display(top_features.to_frame(name="loading_magnitude"))

### Model 4: Gradient Boosting Regressor

In [29]:
from sklearn.ensemble import GradientBoostingRegressor

# Create a smaller tuning sample from the training data
rng = np.random.default_rng(42)

gbr_tune_size = min(50000, len(X_train))   # use up to 50,000 rows for tuning
gbr_tune_indices = rng.choice(len(X_train), size=gbr_tune_size, replace=False)

X_gbr_tune = X_train[gbr_tune_indices]
y_gbr_tune = y_train.iloc[gbr_tune_indices]

In [30]:
# Test several hyperparameter combinations

gbr_param_grid = [
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2, "min_samples_leaf": 5},
    {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 2, "min_samples_leaf": 5},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2, "min_samples_leaf": 5},
    {"n_estimators": 150, "learning_rate": 0.10, "max_depth": 2, "min_samples_leaf": 5},
    {"n_estimators": 200, "learning_rate": 0.10, "max_depth": 3, "min_samples_leaf": 5},
    {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 3, "min_samples_leaf": 10},
]

gbr_results = []

for params in gbr_param_grid:
    # Build one candidate model using the current parameter combination
    gbr_candidate = GradientBoostingRegressor(
        random_state=42,
        **params
    )

    # Fit the model on the tuning sample
    gbr_candidate.fit(X_gbr_tune, y_gbr_tune)

    # Predict on the same tuning sample
    # Ideally, tuning would use a separate validation split but because Gradient Boosting is slower and the dataset is so large, this smaller sampled approach is the practical choice
    y_gbr_tune_pred = gbr_candidate.predict(X_gbr_tune)

    # Evaluate using RMSE and R^2
    gbr_rmse = np.sqrt(mean_squared_error(y_gbr_tune, y_gbr_tune_pred))
    gbr_r2 = r2_score(y_gbr_tune, y_gbr_tune_pred)

    # Save the results so we can compare all models
    gbr_results.append({
        "n_estimators": params["n_estimators"],
        "learning_rate": params["learning_rate"],
        "max_depth": params["max_depth"],
        "min_samples_leaf": params["min_samples_leaf"],
        "tune_rmse": gbr_rmse,
        "tune_r2": gbr_r2
    })

# Turn the results into a DataFrame and sort by the best R^2
gbr_results_df = pd.DataFrame(gbr_results).sort_values(by="tune_r2", ascending=False)

print("Gradient Boosting tuning results:")
display(gbr_results_df)

# Pick the best parameter combination
best_gbr_params = gbr_results_df.iloc[0][["n_estimators", "learning_rate", "max_depth", "min_samples_leaf"]].to_dict()

# Some values may come out as floats because they were stored in a DataFrame,
# so convert the integer-valued hyperparameters back to integers.
best_gbr_params["n_estimators"] = int(best_gbr_params["n_estimators"])
best_gbr_params["max_depth"] = int(best_gbr_params["max_depth"])
best_gbr_params["min_samples_leaf"] = int(best_gbr_params["min_samples_leaf"])

print("Best Gradient Boosting parameters:")
print(best_gbr_params)

Gradient Boosting tuning results:


,n_estimators,learning_rate,max_depth,min_samples_leaf,tune_rmse,tune_r2
4,200,0.10,3,5,0.061410,0.998847
5,150,0.05,3,10,0.124126,0.995290
3,150,0.10,2,5,0.127099,0.995062
2,200,0.05,2,5,0.164193,0.991759
1,150,0.05,2,5,0.194182,0.988474
0,100,0.05,2,5,0.247288,0.981307


Best Gradient Boosting parameters:
{'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 5}


In [31]:
# Train the final Gradient Boosting model on the full training set
gbr_model = GradientBoostingRegressor(
    random_state=42,
    **best_gbr_params
)

gbr_model.fit(X_train, y_train)

GradientBoostingRegressor(min_samples_leaf=5, n_estimators=200, random_state=42)

In [32]:
# Evaluate the final model on both training and test data

gbr_train_eval_size = min(20000, len(X_train))
gbr_train_eval_indices = rng.choice(len(X_train), size=gbr_train_eval_size, replace=False)

X_gbr_train_eval = X_train[gbr_train_eval_indices]
y_gbr_train_eval = y_train.iloc[gbr_train_eval_indices]

# Make predictions
gbr_train_pred = gbr_model.predict(X_gbr_train_eval)
gbr_test_pred = gbr_model.predict(X_test)

# Calculate RMSE and R^2
gbr_train_rmse = np.sqrt(mean_squared_error(y_gbr_train_eval, gbr_train_pred))
gbr_test_rmse = np.sqrt(mean_squared_error(y_test, gbr_test_pred))
gbr_train_r2 = r2_score(y_gbr_train_eval, gbr_train_pred)
gbr_test_r2 = r2_score(y_test, gbr_test_pred)

print("Gradient Boosting Regressor Results:")
print(f"Train RMSE: {gbr_train_rmse:.4f}")
print(f"Test RMSE: {gbr_test_rmse:.4f}")
print(f"Train R^2: {gbr_train_r2:.4f}")
print(f"Test R^2: {gbr_test_r2:.4f}")

Gradient Boosting Regressor Results:
Train RMSE: 0.0606
Test RMSE: 0.0593
Train R^2: 0.9989
Test R^2: 0.9989


In [33]:
# Check for overfitting

gbr_r2_gap = gbr_train_r2 - gbr_test_r2
gbr_rmse_gap = gbr_test_rmse - gbr_train_rmse

print("\nGradient Boosting Overfitting Check")
print(f"R^2 gap (train - test): {gbr_r2_gap:.4f}")
print(f"RMSE gap (test - train): {gbr_rmse_gap:.4f}")

if gbr_r2_gap < 0.03:
    print("This gap is small, so the Gradient Boosting model does not appear to be heavily overfitted.")
else:
    print("This gap is fairly large, so the Gradient Boosting model may be overfitting.")
    print("Possible fixes: reduce max_depth, increase min_samples_leaf, or lower the learning rate.")


Gradient Boosting Overfitting Check
R^2 gap (train - test): -0.0001
RMSE gap (test - train): -0.0013
This gap is small, so the Gradient Boosting model does not appear to be heavily overfitted.


In [34]:
# Show the most important features

gbr_feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": gbr_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nTop 15 most important features for Gradient Boosting:")
display(gbr_feature_importance_df.head(15))


Top 15 most important features for Gradient Boosting:


,feature,importance
20,daily_active_minutes_instagram,0.892300
40,average_session_length_minutes,0.053712
22,posts_created_per_week,0.025524
25,likes_given_per_day,0.014154
26,comments_written_per_day,0.014083
0,age,0.000147
34,time_on_reels_per_day,0.000065
23,reels_watched_per_day,0.000006
32,time_on_explore_per_day,0.000006
31,time_on_feed_per_day,0.000004


### Analysis

The Gradient Boosting Regressor performed extremely well on this dataset, achieving a test R² of 0.9989 and a test RMSE of 0.0593, indicating that the model explains nearly all of the variance in Instagram engagement. The training and testing performance are nearly identical, with an R² gap of -0.0001 and RMSE gap of -0.0013, suggesting that the model generalizes very well and does not suffer from overfitting. This confirms that the model is capturing meaningful patterns in the data rather than memorizing noise.

The best-performing model used:
- 200 estimators → more trees improve learning capacity
- learning rate of 0.1 → balances speed and accuracy
- max depth of 3 → allows moderate complexity without overfitting
- min samples leaf of 5 → prevents the model from fitting noise

These choices reflect a balance between model flexibility and generalization, which is why the overfitting check shows such a small gap.

Additionally, evaluation on a random subset of the training data (20,000 rows) was used instead of the full training set. This was done to significantly reduce computation time while still providing an accurate estimate of training performance.

The feature importance results provide strong insight into what drives engagement. The model shows that:
- daily_active_minutes_instagram (~89%) dominates the model, meaning engagement is almost entirely driven by time spent on the app
- Secondary features like average session length, posts created, likes, and comments have much smaller but still meaningful contributions
- Many demographic variables (e.g., country, employment, education) have near-zero importance, suggesting they do not strongly influence engagement in this dataset